# Pretrain Commutative Transformer Encoder

Load the shared unlabeled pretraining dataset and save commutative transformer encoder weights for downstream transformer experiments.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from src.ml import CommutativeTransformerClassifier, CommutativeTransformerConfig, LossWeightConfig, OptimizationConfig
from src.tensor_utils import load_unlabeled_tensor_dataset


In [ ]:
# User inputs

unlabeled_dataset_path = Path(".dataset_cache/unlabeled_active_high_mid_low_t20_z5_y64_x64.pt")
pretrained_encoder_path = Path("artifacts/pretrained_commutative_transformer/encoder_state.pt")

model_config = CommutativeTransformerConfig(
    spatial_patch_size_st=(1, 32, 32),
    spatial_patch_size_ts=(1, 32, 32),
    temporal_patch_size_ts=2,
    embed_dim=32,
    num_heads=2,
    mlp_ratio=2.0,
    dropout=0.3,
    attention_dropout=0.1,
    st_spatial_depth=1,
    st_temporal_depth=1,
    ts_temporal_depth=1,
    ts_spatial_depth=1,
    embedding_dim=16,
    num_prototypes=8,
)
optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=75,
    learning_rate=1e-4,
    weight_decay=3e-3,
    early_stopping_patience=8,
    early_stopping_min_delta=0.0,
    scheduler_patience=3,
    scheduler_factor=0.7,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    consistency_weight=1.0,
    feature_weight=0.05,
    prototype_temperature=0.1,
)

In [ ]:
unlabeled_dataset = load_unlabeled_tensor_dataset(unlabeled_dataset_path)
unlabeled_dataset["tensors"].shape, unlabeled_dataset["metadata"].shape

In [ ]:
model = CommutativeTransformerClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
model.pretrain(unlabeled_dataset["tensors"])
pretrained_encoder_path = model.save_pretrained_encoder(pretrained_encoder_path)
pretrained_encoder_path

In [ ]:
model.pretrain_history_.tail()